In [199]:
import numpy as np
import pandas as pd
from joblib import load
import os

In [200]:
os.chdir(os.getcwd())

# Defining Fsp end-members

In [201]:
# chemical format SiO2, TiO2, Al2O3, FeO, MnO, MgO, CaO, Na2O, K2O, P2O5, CO2
an = np.array([2,0,1,0,0,0,1,0,0,0,0])
ab = np.array([3,0,0.5,0,0,0,0,1,0,0,0])
ort = np.array([3,0,0.5,0,0,0,0,0,1,0,0])

an = 100 * an / sum(an)
ab = 100 * ab / sum(ab)
ort = 100 * ort / sum(ort)

In [240]:
def make_pl(interval = 0.1):
    an = np.array([2,0,1,0,0,0,1,0,0,0,0])
    ab = np.array([3,0,0.5,0,0,0,0,1,0,0,0])
    an = 100 * an / sum(an)
    ab = 100 * ab / sum(ab)
    pl = np.array([])
    for i in np.arange(0,1 + interval,interval):
        s = round(i,2)
        comp = (an * s) + (ab * (1 - s))
        # print(comp)
        if (i == 0):
            pl = comp
        else:
            pl = np.vstack((pl,comp))
    return(pl)

def make_fsp(interval = 0.1):
    pl = make_pl(interval)
    ort = np.array([3,0,0.5,0,0,0,0,0,1,0,0])
    ort = 100 * ort / sum(ort)
    c = 0
    fsp = np.array([])
    for i in np.arange(0,np.shape(pl)[0]):
        for j in np.arange(0,1 + interval, interval):
            comp = ort * j + (pl[i] * (1 - j))
            c = c + 1
            if (c == 1):
                fsp = comp
            else:
                fsp = np.vstack((fsp,comp))
    
    np.shape(fsp)
    return(fsp)

In [418]:
fsp = make_fsp(0.01)
fsp = pd.DataFrame(fsp, columns = ["SiO2", "TiO2", "Al2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O", "P2O5", "CO2"])

# Load model (and other files) and make predictions

In [558]:
mdl = "RF"
combination = "C4"

In [559]:
if (mdl == "KNN"):
    path = r"C:\Users\naik3\Documents\Research\Mineral identifier ann IISER Mohali\Additional machine learning algos\KNN\\"
if (mdl == "SVM"):
    path = r"C:\Users\naik3\Documents\Research\Mineral identifier ann IISER Mohali\Additional machine learning algos\SVM\\"
if (mdl == "RF"):
    path = r"C:\Users\naik3\Documents\Research\Mineral identifier ann IISER Mohali\Additional machine learning algos\RF\\"

if (combination != "C4"):
    model = mdl + "_" + combination + ".mdl"
    labeler = "Labeler"+ "_" + mdl + "_" + combination + ".lbl"
    if (mdl != "RF"):
        scaler = "Scaler" + "_" + mdl + "_" + combination + ".scl"
else:
    model = mdl + "_" + combination + "_5_components" + ".mdl"
    labeler = "Labeler"+ "_" + mdl + "_" + combination + "_5_components" + ".lbl"
    pc = "PCA_" + mdl + "_" + combination + "_5_components" + ".pc"
    if (mdl != "RF"):
        scaler = "Scaler" + "_" + mdl + "_" + combination + "_5_components" + ".scl"

print(model, scaler, labeler, pc)

RF_C4_5_components.mdl Scaler_RF_C1.scl Labeler_RF_C4_5_components.lbl PCA_RF_C4_5_components.pc


In [560]:
model = load(path + model)
if (mdl != "RF"):
    scaler = load(path + scaler)
labeler = load(path + labeler)

if (combination == "C4"):
    pc_model = load(path + pc)
    
c = fsp.columns

if ("PredMin" in c):
    fsp.pop("PredMin")
if ("Total" in c):
    fsp.pop("Total")

In [561]:
fsp1 = fsp.copy()

if (combination == "C2"):
    fsp1["M"] = fsp1["FeO"] + fsp1["MnO"] + fsp1["MgO"]
    fsp1.drop(columns=["FeO","MnO","MgO"])
    fsp1 = fsp1[["SiO2", "TiO2", "Al2O3", "M", "CaO", "Na2O", "K2O", "P2O5", "CO2"]]
    
if (combination == "C3"):
    fsp1["M"] = fsp1["FeO"] + fsp1["MnO"] + fsp1["MgO"]
    fsp1["A"] = fsp1["Na2O"] + fsp1["K2O"]
    fsp1.drop(columns=["FeO","MnO","MgO","Na2O","K2O"])
    fsp1 = fsp1[["SiO2", "TiO2", "Al2O3", "M", "CaO", "A", "P2O5", "CO2"]]

if (combination == "C4"):
    fsp1 = fsp1[["SiO2", "TiO2", "Al2O3", "FeO", "MnO", "MgO", "CaO", "Na2O", "K2O", "P2O5", "CO2"]]
    if (mdl != "RF"):
        fsp1 = scaler.transform(fsp1)
    data_scaled = pc_model.transform(fsp1)

fsp1

C:\Users\naik3\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:458: UserWarning: X has feature names, but PCA was fitted without feature names
  warnings.warn(


,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,CO2
0,66.666667,0.0,11.111111,0.0,0.0,0.0,0.00,22.222222,0.000000,0.0,0.0
1,66.666667,0.0,11.111111,0.0,0.0,0.0,0.00,22.000000,0.222222,0.0,0.0
2,66.666667,0.0,11.111111,0.0,0.0,0.0,0.00,21.777778,0.444444,0.0,0.0
3,66.666667,0.0,11.111111,0.0,0.0,0.0,0.00,21.555556,0.666667,0.0,0.0
4,66.666667,0.0,11.111111,0.0,0.0,0.0,0.00,21.333333,0.888889,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
10196,66.000000,0.0,11.666667,0.0,0.0,0.0,1.00,0.000000,21.333333,0.0,0.0
10197,66.166667,0.0,11.527778,0.0,0.0,0.0,0.75,0.000000,21.555556,0.0,0.0
10198,66.333333,0.0,11.388889,0.0,0.0,0.0,0.50,0.000000,21.777778,0.0,0.0
10199,66.500000,0.0,11.250000,0.0,0.0,0.0,0.25,0.000000,22.000000,0.0,0.0


In [562]:
np.shape(data_scaled)

(10201, 5)

In [563]:
if (combination != "C4"):
    if (mdl != "RF"):
        data_scaled = scaler.transform(fsp1)
    elif (mdl == "RF"):
        data_scaled = fsp1.copy()

In [564]:
data_scaled

array([[-22.41350989,  14.58011719,  21.01960831, -19.06741457,
         -4.75058517],
       [-22.41408505,  14.5813822 ,  21.01692422, -19.06988423,
         -4.74977417],
       [-22.4146602 ,  14.58264722,  21.01424012, -19.07235388,
         -4.74896317],
       ...,
       [-22.09926374,  14.89080327,  20.85207542, -18.88885574,
         -4.59081309],
       [-22.2851445 ,  14.79871103,  20.80163702, -19.1016181 ,
         -4.63014906],
       [-22.47102526,  14.70661878,  20.75119861, -19.31438046,
         -4.66948502]])

In [565]:
pred = model.predict(data_scaled)
predictions = labeler.inverse_transform(pred)

In [566]:
predictions[predictions != "Fsp"]

array([], dtype=object)

In [567]:
x = model.predict_proba(data_scaled)
fsp["Total"] = fsp.sum(axis = 1)
fsp["PredMin"] = predictions
fsp_prob = pd.DataFrame(x, columns = labeler.classes_, index = fsp.index)

In [568]:
fsp_final = pd.concat([fsp,fsp_prob], axis = 1)

In [569]:
fsp_final

,SiO2,TiO2,Al2O3,FeO,MnO,MgO,CaO,Na2O,K2O,P2O5,...,Ilm,Mag/Hem,Ms,Ol,Px,Qz,Rt,Spl,St,Ttn
0,66.666667,0.0,11.111111,0.0,0.0,0.0,0.00,22.222222,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0
1,66.666667,0.0,11.111111,0.0,0.0,0.0,0.00,22.000000,0.222222,0.0,...,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0
2,66.666667,0.0,11.111111,0.0,0.0,0.0,0.00,21.777778,0.444444,0.0,...,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0
3,66.666667,0.0,11.111111,0.0,0.0,0.0,0.00,21.555556,0.666667,0.0,...,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0
4,66.666667,0.0,11.111111,0.0,0.0,0.0,0.00,21.333333,0.888889,0.0,...,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10196,66.000000,0.0,11.666667,0.0,0.0,0.0,1.00,0.000000,21.333333,0.0,...,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0
10197,66.166667,0.0,11.527778,0.0,0.0,0.0,0.75,0.000000,21.555556,0.0,...,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0
10198,66.333333,0.0,11.388889,0.0,0.0,0.0,0.50,0.000000,21.777778,0.0,...,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0
10199,66.500000,0.0,11.250000,0.0,0.0,0.0,0.25,0.000000,22.000000,0.0,...,0.0,0.0,0.0,0.0,0.1,0.0,0.0,0.0,0.0,0.0


In [570]:
fsp_final.Total

0        100.0
1        100.0
2        100.0
3        100.0
4        100.0
         ...  
10196    100.0
10197    100.0
10198    100.0
10199    100.0
10200    100.0
Name: Total, Length: 10201, dtype: float64

In [571]:
output_file = "Fsp_prediction_for_" + mdl + "_" + combination + ".csv"
output_file

'Fsp_prediction_for_RF_C4.csv'

In [572]:
fsp_final.to_csv(output_file)